# Setup:

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [4]:
coffeeURL = "data/02-processed/coffee_prices_clean.csv"
coffee_df = pd.read_csv(coffeeURL)

oilURL = "data/02-processed/oil-prices-clean.csv"
oil_df = pd.read_csv(oilURL)

cpiURL = "data/02-processed/inflation_clean.csv"
cpi_df = pd.read_csv(cpiURL)


In [5]:
coffee_df.head()

,date,value
0,1973-08-20,0.6735
1,1973-08-21,0.6710
2,1973-08-22,0.6580
3,1973-08-23,0.6675
4,1973-08-24,0.6660


In [6]:
oil_df.head()

,date,price,percentChange,change,is_outlier
0,1983-04-01 00:00:00+00:00,30.63,4.646,1.36,False
1,1983-05-01 00:00:00+00:00,30.25,-1.241,-0.38,False
2,1983-06-01 00:00:00+00:00,31.38,3.736,1.13,False
3,1983-07-01 00:00:00+00:00,32.00,1.976,0.62,False
4,1983-08-01 00:00:00+00:00,31.59,-1.281,-0.41,False


In [7]:
cpi_df.head()

,Unnamed: 0,cpi
0,1913-01-01,9.8
1,1913-01-02,9.8
2,1913-01-03,9.8
3,1913-01-04,9.8
4,1913-01-05,9.8


### Setting uniform date format and type

In [8]:
# rename date column of cpi to date
cpi_df = cpi_df.rename(columns={'Unnamed: 0': 'date'})

# remove time component from oil date format 
oil_df['date'] = pd.to_datetime(oil_df['date']).dt.tz_localize(None)
# makes sure all date types are datetime64[ns]
coffee_df['date'] = pd.to_datetime(coffee_df['date'])
cpi_df['date'] = pd.to_datetime(cpi_df['date'])

# check date types
print(coffee_df['date'].dtype)
print(oil_df['date'].dtype)
print(cpi_df['date'].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]


### Setting up column

In [9]:
coffee_df = coffee_df.rename(columns={'value': 'coffeeCost'})
oil_df = oil_df.rename(columns={'price': 'oilCost'})

print(coffee_df.head())
print(oil_df.head())

        date  coffeeCost
0 1973-08-20      0.6735
1 1973-08-21      0.6710
2 1973-08-22      0.6580
3 1973-08-23      0.6675
4 1973-08-24      0.6660
        date  oilCost  percentChange  change  is_outlier
0 1983-04-01    30.63          4.646    1.36       False
1 1983-05-01    30.25         -1.241   -0.38       False
2 1983-06-01    31.38          3.736    1.13       False
3 1983-07-01    32.00          1.976    0.62       False
4 1983-08-01    31.59         -1.281   -0.41       False


### Filtering out data outside date window

In [10]:
coffee_window = coffee_df[(coffee_df['date'].dt.year >= 2009) & 
                          (coffee_df['date'].dt.year <= 2021)]
oil_window = oil_df[(oil_df['date'].dt.year >= 2009) & 
                          (oil_df['date'].dt.year <= 2021)]
cpi_window = cpi_df[(cpi_df['date'].dt.year >= 2009) & 
                          (cpi_df['date'].dt.year <= 2021)]

print(coffee_window.head())
print(oil_window.head())
print(cpi_window.head())

           date  coffeeCost
8839 2009-01-02      1.1090
8840 2009-01-05      1.0770
8841 2009-01-06      1.1610
8842 2009-01-07      1.1420
8843 2009-01-08      1.1345
          date  oilCost  percentChange  change  is_outlier
309 2009-01-01    41.68         -6.547   -2.92       False
310 2009-02-01    44.76          7.390    3.08       False
311 2009-03-01    49.66         10.947    4.90       False
312 2009-04-01    51.12          2.940    1.46       False
313 2009-05-01    66.31         29.714   15.19       False
            date      cpi
35064 2009-01-01  211.143
35065 2009-01-02  211.143
35066 2009-01-03  211.143
35067 2009-01-04  211.143
35068 2009-01-05  211.143


### Merging data

In [11]:
merged_df = coffee_window.merge(oil_window[['date', 'oilCost']], on='date', how='outer').merge(cpi_window[['date', 'cpi']], on='date', how='outer')

merged_df = merged_df.sort_values('date').reset_index(drop=True)
print(merged_df.head())
print(f"Total rows: {len(merged_df)}")

        date  coffeeCost  oilCost      cpi
0 2009-01-01         NaN    41.68  211.143
1 2009-01-02       1.109      NaN  211.143
2 2009-01-03         NaN      NaN  211.143
3 2009-01-04         NaN      NaN  211.143
4 2009-01-05       1.077      NaN  211.143
Total rows: 4748


In [12]:
# CLEANING: delete rows that neither have a price for coffee or oil
print("=== BEFORE CLEANING ===")
print(f"Total rows: {len(merged_df)}")
print("\nMissing values per column:")
print(merged_df[['coffeeCost', 'oilCost', 'cpi']].isnull().sum())
print("\nRows where both coffeeCost and oilCost are missing:")
both_missing = merged_df[merged_df['coffeeCost'].isna() & merged_df['oilCost'].isna()]
print(f"Count: {len(both_missing)}")
if len(both_missing) > 0:
    print("Sample of rows missing both:")
    print(both_missing.head())

# Clean the data
cleaned_df = merged_df.dropna(subset=['coffeeCost', 'oilCost'], how='all')

# After cleaning
print("\n=== AFTER CLEANING ===")
print(f"Total rows: {len(cleaned_df)}")
print("\nMissing values per column:")
print(cleaned_df[['coffeeCost', 'oilCost', 'cpi']].isnull().sum())


=== BEFORE CLEANING ===
Total rows: 4748

Missing values per column:
coffeeCost    1455
oilCost       4592
cpi              0
dtype: int64

Rows where both coffeeCost and oilCost are missing:
Count: 1400
Sample of rows missing both:
         date  coffeeCost  oilCost      cpi
2  2009-01-03         NaN      NaN  211.143
3  2009-01-04         NaN      NaN  211.143
9  2009-01-10         NaN      NaN  211.143
10 2009-01-11         NaN      NaN  211.143
16 2009-01-17         NaN      NaN  211.143

=== AFTER CLEANING ===
Total rows: 3348

Missing values per column:
coffeeCost      55
oilCost       3192
cpi              0
dtype: int64


In [13]:
cleaned_df.head(10)

,date,coffeeCost,oilCost,cpi
0,2009-01-01,NaN,41.68,211.143
1,2009-01-02,1.1090,NaN,211.143
4,2009-01-05,1.0770,NaN,211.143
5,2009-01-06,1.1610,NaN,211.143
6,2009-01-07,1.1420,NaN,211.143
7,2009-01-08,1.1345,NaN,211.143
8,2009-01-09,1.1690,NaN,211.143
11,2009-01-12,1.1450,NaN,211.143
12,2009-01-13,1.1475,NaN,211.143
13,2009-01-14,1.1465,NaN,211.143


In [14]:
#show the first rows where oil has prices
first_oil_rows = cleaned_df[cleaned_df['oilCost'].notna()].head(10)
print(first_oil_rows)

#fill oilCost to match number of occcurences as coffee
filled_df = cleaned_df.copy()
filled_df['oilCost'] = cleaned_df['oilCost'].ffill()

#drop rows without coffee prices
filled_df = filled_df.dropna(subset=['coffeeCost'])

print(filled_df.head(10))
print(filled_df.tail(10))

          date  coffeeCost  oilCost      cpi
0   2009-01-01         NaN    41.68  211.143
31  2009-02-01         NaN    44.76  212.193
59  2009-03-01         NaN    49.66  212.709
90  2009-04-01      1.1450    51.12  213.240
120 2009-05-01      1.2040    66.31  213.856
151 2009-06-01      1.4225    69.89  215.693
181 2009-07-01      1.1905    69.45  215.351
212 2009-08-01         NaN    69.96  215.834
243 2009-09-01      1.2010    70.61  215.969
273 2009-10-01      1.2670    77.00  216.177
         date  coffeeCost  oilCost      cpi
1  2009-01-02      1.1090    41.68  211.143
4  2009-01-05      1.0770    41.68  211.143
5  2009-01-06      1.1610    41.68  211.143
6  2009-01-07      1.1420    41.68  211.143
7  2009-01-08      1.1345    41.68  211.143
8  2009-01-09      1.1690    41.68  211.143
11 2009-01-12      1.1450    41.68  211.143
12 2009-01-13      1.1475    41.68  211.143
13 2009-01-14      1.1465    41.68  211.143
14 2009-01-15      1.1390    41.68  211.143
           date  coff

In [15]:
cleaned_df[['date','oilCost']].dropna().head(20)

,date,oilCost
0,2009-01-01,41.68
31,2009-02-01,44.76
59,2009-03-01,49.66
90,2009-04-01,51.12
120,2009-05-01,66.31
151,2009-06-01,69.89
181,2009-07-01,69.45
212,2009-08-01,69.96
243,2009-09-01,70.61
273,2009-10-01,77.00


# Price Deflation

For Reference, example from [US Inflation Calculator](https://www.usinflationcalculator.com/inflation/coffee-prices-by-year-and-adjust-for-inflation/)
$$
\text{1980 Coffee Price} \times \left( \frac{\text{2022 CPI for Coffee}}{\text{1980 CPI for Coffee}} \right) = \text{Adjusted Coffee Price in 2022 Dollars}
$$

Using actual numbers: 

$$3.14 \times \left(\frac{240.298}{116.9} \right) = 6.45$$

In [16]:
cpi2022 = 278.802
all_df = filled_df.copy()
all_df['coffeeD'] = filled_df['coffeeCost'] * (cpi2022 / filled_df['cpi'])
all_df['oilD'] = filled_df['oilCost'] * (cpi2022 / filled_df['cpi'])

print(all_df.head(10))
print(all_df.tail(10))

         date  coffeeCost  oilCost      cpi   coffeeD       oilD
1  2009-01-02      1.1090    41.68  211.143  1.464370  55.036006
4  2009-01-05      1.0770    41.68  211.143  1.422116  55.036006
5  2009-01-06      1.1610    41.68  211.143  1.533033  55.036006
6  2009-01-07      1.1420    41.68  211.143  1.507944  55.036006
7  2009-01-08      1.1345    41.68  211.143  1.498041  55.036006
8  2009-01-09      1.1690    41.68  211.143  1.543596  55.036006
11 2009-01-12      1.1450    41.68  211.143  1.511906  55.036006
12 2009-01-13      1.1475    41.68  211.143  1.515207  55.036006
13 2009-01-14      1.1465    41.68  211.143  1.513886  55.036006
14 2009-01-15      1.1390    41.68  211.143  1.503983  55.036006
           date  coffeeCost  oilCost      cpi  coffeeD   oilD
4736 2021-12-20      2.2410    74.88  278.802   2.2410  74.88
4737 2021-12-21      2.2825    74.88  278.802   2.2825  74.88
4738 2021-12-22      2.3355    74.88  278.802   2.3355  74.88
4739 2021-12-23      2.3120    74.88 

In [ ]:
# Save to data processed folder
#output_path = "data/02-processed/deflated-prices.csv"
#all_df.to_csv(output_path, index=False)